# 04 Adult/chick 日期代理比较

主分析为 `adult_proxy_vs_chick_proxy`；`early_vs_possible_chick_0716_p080` 仅用于日期边界敏感性分析。统计单位为来源录音：命中率按录音小时数标准化，声学特征先在录音内取中位数。日期阶段是代理变量，结果是关联性而非因果性；sigmoid 模型分数未校准，不能解释为真实生物学概率。

In [6]:
from pathlib import Path
import os
import sys

def find_project_root(start=None):
    path = Path(start or Path.cwd()).resolve()
    for candidate in [path, *path.parents]:
        if (candidate / "project_config.py").is_file():
            return candidate
    raise FileNotFoundError("Run this notebook from inside the project repository.")

PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from project_config import WORKSPACE_DIR

os.environ.setdefault("MPLBACKEND", "Agg")
os.environ.setdefault("MPLCONFIGDIR", str(WORKSPACE_DIR / "matplotlib_cache"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

pd.set_option("display.max_columns", 60)
pd.set_option("display.max_colwidth", 120)

def require_columns(frame, required, name):
    missing = sorted(set(required) - set(frame.columns))
    if missing:
        raise ValueError(f"{name} is missing required columns: {missing}")

def save_figure(fig, stem):
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / f"{stem}.png", dpi=300, bbox_inches="tight")
    fig.savefig(OUTPUT_DIR / f"{stem}.pdf", bbox_inches="tight")
    plt.close(fig)

def write_latex(frame, name, **kwargs):
    (OUTPUT_DIR / name).write_text(frame.to_latex(index=False, **kwargs), encoding="utf-8")

from scipy.stats import mannwhitneyu
from project_config import COMP_CHICK_ADULT_DIR

ANALYSIS_THRESHOLD = 0.80
BOOTSTRAP_ITERATIONS = 5000
RANDOM_SEED = 2021
MAIN_NAME = "adult_proxy_vs_chick_proxy"
SENSITIVITY_NAME = "early_vs_possible_chick_0716_p080"
OUTPUT_DIR = WORKSPACE_DIR / "paper_analysis" / "04_adult_chick_proxy"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FEATURES = ["rms", "dbfs", "peak_frequency_hz", "spectral_centroid_hz", "spectral_bandwidth_hz",
            "rolloff_85_hz", "spectral_flatness", "spectral_entropy", "zero_crossing_rate"]

def input_paths(name):
    root = COMP_CHICK_ADULT_DIR / name
    return {key: root / filename for key, filename in {
        "recording": "recording_summary.csv", "yield": "threshold_yield.csv",
        "windows": "analysis_windows.csv", "all_windows": "scan/all_windows.csv"
    }.items()}

for name in [MAIN_NAME, SENSITIVITY_NAME]:
    missing = [str(path) for path in input_paths(name).values() if not path.is_file()]
    if missing:
        raise FileNotFoundError(f"Missing {name} inputs: {missing}")
print(f"Output directory: {OUTPUT_DIR}")

Output directory: /home/zenith/Projects/strom_petrels/workspace/paper_analysis/04_adult_chick_proxy


In [7]:
def load_analysis(name):
    paths = input_paths(name)
    recording = pd.read_csv(paths["recording"])
    threshold_yield = pd.read_csv(paths["yield"])
    high_windows = pd.read_csv(paths["windows"], usecols=["recording", "recording_date", "phase", "probability"])
    all_windows = pd.read_csv(paths["all_windows"], usecols=["recording", "probability"])
    require_columns(recording, {"recording", "recording_date", "phase", "recording_duration_seconds",
                                "eligible_window_count", "high_confidence_window_count", "high_confidence_hits_per_hour",
                                *FEATURES}, f"{name} recording summary")
    require_columns(threshold_yield, {"threshold", "phase", "window_count", "recording_count"}, f"{name} threshold yield")
    phase_map = recording.set_index("recording")["phase"]
    all_windows["phase"] = all_windows["recording"].map(phase_map)
    if all_windows["phase"].isna().any():
        raise AssertionError(f"Unmapped recordings in {name} all_windows.csv")
    if not all_windows["probability"].between(0, 1, inclusive="both").all():
        raise AssertionError(f"Scores outside [0, 1] in {name}")
    return recording, threshold_yield, high_windows, all_windows

main_recording, main_yield, main_high, main_all = load_analysis(MAIN_NAME)
sens_recording, sens_yield, sens_high, sens_all = load_analysis(SENSITIVITY_NAME)

module_summary = pd.DataFrame([
    {"analysis": name, "recordings": rec.recording.nunique(), "adult_proxy_recordings": int((rec.phase == "adult_proxy").sum()),
     "chick_proxy_recordings": int((rec.phase == "chick_proxy").sum()),
     "high_confidence_windows": int(rec.high_confidence_window_count.sum()),
     "recordings_with_hits": int((rec.high_confidence_window_count > 0).sum())}
    for name, rec in [(MAIN_NAME, main_recording), (SENSITIVITY_NAME, sens_recording)]
])
module_summary.to_csv(OUTPUT_DIR / "analysis_summary.csv", index=False)
write_latex(module_summary, "analysis_summary.tex")
display(module_summary)

,analysis,recordings,adult_proxy_recordings,chick_proxy_recordings,high_confidence_windows,recordings_with_hits
0,adult_proxy_vs_chick_proxy,38,26,12,15668,10
1,early_vs_possible_chick_0716_p080,51,26,25,17520,23


## 录音级活动、阈值产量与分数分布

日期趋势展示每小时高置信窗口数。阈值产量是描述性敏感性分析；主阈值固定为 0.80，不根据当前数据优化。

In [3]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
for phase, group in main_recording.groupby("phase"):
    ordered = group.assign(date=pd.to_datetime(group.recording_date)).sort_values("date")
    axes[0, 0].plot(ordered.date, ordered.high_confidence_hits_per_hour, marker="o", label=phase)
axes[0, 0].set(title="Recording-level high-confidence activity", xlabel="Recording date", ylabel="Hits per recording hour")
axes[0, 0].legend()

for phase, group in main_yield.groupby("phase"):
    axes[0, 1].plot(group.threshold, group.window_count, marker="o", label=phase)
axes[0, 1].axvline(ANALYSIS_THRESHOLD, color="black", linestyle="--")
axes[0, 1].set(title="Threshold yield: main date proxy", xlabel="Sigmoid score threshold", ylabel="Windows")
axes[0, 1].legend()

for phase, group in sens_recording.groupby("phase"):
    ordered = group.assign(date=pd.to_datetime(group.recording_date)).sort_values("date")
    axes[1, 0].plot(ordered.date, ordered.high_confidence_hits_per_hour, marker="o", label=phase)
axes[1, 0].set(title="Date-boundary sensitivity", xlabel="Recording date", ylabel="Hits per recording hour")
axes[1, 0].legend()

for phase, group in main_all.groupby("phase"):
    axes[1, 1].hist(group.probability, bins=50, density=True, histtype="step", linewidth=1.7, label=phase)
axes[1, 1].axvline(ANALYSIS_THRESHOLD, color="black", linestyle="--")
axes[1, 1].set(title="Window score distribution (descriptive)", xlabel="Uncalibrated sigmoid model score", ylabel="Density", yscale="log")
axes[1, 1].legend()
save_figure(fig, "activity_yield_probability")

## 录音级声学特征效应量

每个特征先取录音级中位数，再比较两个日期代理组。表格报告 chick proxy − adult proxy 的中位数差、录音级 bootstrap 95% CI、Cliff's delta、双侧 Mann–Whitney U 检验与 BH-FDR。窗口级数值不用于推断。

In [4]:
def cliffs_delta(adult, chick):
    comparisons = np.subtract.outer(chick, adult)
    return (np.sum(comparisons > 0) - np.sum(comparisons < 0)) / comparisons.size

def bh_fdr(values):
    values = np.asarray(values, dtype=float)
    result = np.full(values.shape, np.nan)
    valid = np.flatnonzero(np.isfinite(values))
    if not len(valid):
        return result
    order = valid[np.argsort(values[valid])]
    adjusted = values[order] * len(order) / np.arange(1, len(order) + 1)
    adjusted = np.minimum.accumulate(adjusted[::-1])[::-1]
    result[order] = np.minimum(adjusted, 1.0)
    return result

def recording_feature_effects(recording, analysis_name, seed):
    rng = np.random.default_rng(seed)
    rows = []
    for feature in FEATURES:
        adult = recording.loc[recording.phase == "adult_proxy", feature].dropna().to_numpy(float)
        chick = recording.loc[recording.phase == "chick_proxy", feature].dropna().to_numpy(float)
        if len(adult) < 2 or len(chick) < 2:
            rows.append({"analysis": analysis_name, "feature": feature, "adult_n": len(adult), "chick_n": len(chick)})
            continue
        observed = np.median(chick) - np.median(adult)
        draws = np.empty(BOOTSTRAP_ITERATIONS)
        for index in range(BOOTSTRAP_ITERATIONS):
            draws[index] = np.median(rng.choice(chick, len(chick), replace=True)) - np.median(rng.choice(adult, len(adult), replace=True))
        test = mannwhitneyu(chick, adult, alternative="two-sided")
        low, high = np.quantile(draws, [0.025, 0.975])
        rows.append({"analysis": analysis_name, "feature": feature, "adult_n": len(adult), "chick_n": len(chick),
                     "adult_median": np.median(adult), "chick_median": np.median(chick),
                     "median_difference_chick_minus_adult": observed, "bootstrap_ci95_low": low,
                     "bootstrap_ci95_high": high, "cliffs_delta_chick_minus_adult": cliffs_delta(adult, chick),
                     "mann_whitney_u": test.statistic, "p_value": test.pvalue})
    result = pd.DataFrame(rows)
    result["p_value_bh_fdr"] = bh_fdr(result.get("p_value", pd.Series(np.nan, index=result.index)))
    return result

effects = pd.concat([
    recording_feature_effects(main_recording, MAIN_NAME, RANDOM_SEED),
    recording_feature_effects(sens_recording, SENSITIVITY_NAME, RANDOM_SEED + 1),
], ignore_index=True)
effects.to_csv(OUTPUT_DIR / "recording_level_feature_effects.csv", index=False)
write_latex(effects[effects.analysis == MAIN_NAME], "recording_level_feature_effects.tex", float_format="%.4g")
display(effects)

,analysis,feature,adult_n,chick_n,adult_median,chick_median,median_difference_chick_minus_adult,bootstrap_ci95_low,bootstrap_ci95_high,cliffs_delta_chick_minus_adult,mann_whitney_u,p_value,p_value_bh_fdr
0,adult_proxy_vs_chick_proxy,rms,7,3,0.015148,0.004822,-0.010326,-0.018680,0.002968,-0.142857,9.0,0.833333,0.937500
1,adult_proxy_vs_chick_proxy,dbfs,7,3,-36.392629,-46.335326,-9.942697,-15.049686,5.235485,-0.142857,9.0,0.833333,0.937500
2,adult_proxy_vs_chick_proxy,peak_frequency_hz,7,3,0.000000,1617.187500,1617.187500,0.000000,1640.625000,0.904762,20.0,0.026486,0.238373
3,adult_proxy_vs_chick_proxy,spectral_centroid_hz,7,3,1210.411957,1837.984081,627.572125,-768.445044,1363.497533,0.619048,17.0,0.183333,0.412500
4,adult_proxy_vs_chick_proxy,spectral_bandwidth_hz,7,3,1988.605965,2542.800048,554.194083,-321.152328,1067.734820,0.714286,18.0,0.116667,0.412500
5,adult_proxy_vs_chick_proxy,rolloff_85_hz,7,3,2089.731340,2670.947617,581.216277,-1596.110612,1907.036871,0.428571,15.0,0.383333,0.690000
6,adult_proxy_vs_chick_proxy,spectral_flatness,7,3,0.000085,0.000254,0.000169,-0.000087,0.000403,0.619048,17.0,0.183333,0.412500
7,adult_proxy_vs_chick_proxy,spectral_entropy,7,3,0.334445,0.321729,-0.012716,-0.233498,0.329314,-0.047619,10.0,1.000000,1.000000
8,adult_proxy_vs_chick_proxy,zero_crossing_rate,7,3,0.018005,0.038065,0.020060,-0.031656,0.053546,0.333333,14.0,0.516667,0.775000
9,early_vs_possible_chick_0716_p080,rms,7,16,0.015148,0.004417,-0.010731,-0.019589,0.010953,-0.250000,42.0,0.375955,0.483371


In [5]:
main_effects = effects[effects.analysis == MAIN_NAME].dropna(subset=["cliffs_delta_chick_minus_adult"])
sens_effects = effects[effects.analysis == SENSITIVITY_NAME].dropna(subset=["cliffs_delta_chick_minus_adult"])
fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharex=True)
for ax, table, title in [(axes[0], main_effects, "Main date proxy"), (axes[1], sens_effects, "Date-boundary sensitivity")]:
    positions = np.arange(len(table))
    ax.scatter(table.cliffs_delta_chick_minus_adult, positions, color="#4c78a8")
    ax.axvline(0, color="gray", linestyle="--")
    ax.set(yticks=positions, yticklabels=table.feature, xlabel="Cliff's delta (chick proxy − adult proxy)", title=title, xlim=(-1.05, 1.05))
save_figure(fig, "recording_level_feature_effects")

print("Inference is recording-level. Date groups are proxies; model scores are uncalibrated and non-causal.")

Inference is recording-level. Date groups are proxies; model scores are uncalibrated and non-causal.
